In [5]:
import requests
import json
import time

GAMMA = "https://gamma-api.polymarket.com"

# Core conflict tag IDs
CONFLICT_TAGS = [
    100265,  # Geopolitics
    79,      # war
    502,     # conflict
    1346,    # international conflicts
    464,     # Military Actions
    193,     # military
    593,     # invasion
    863,     # warfare
    1308,    # military invasion
    334,     # ceasefire
    335,     # peace deals
    415,     # peace deal
    721,     # peace talks
    97,      # peace agreements
    612,     # peace negotiations
    103027,  # Ukraine Peace Deal
    102620,  # Sanctions
    471,     # terrorism
    101409,  # coup
    69,      # strike
    100674,  # missile
    103089,  # troops
    101872,  # drone strike
    102107,  # houthi
    582,     # houthis
    103074,  # israel strike iran
    96,      # Ukraine
    78,      # Iran
    180,     # Israel
    61,      # Gaza
    303,     # China
    867,     # taiwan
    95,      # russia
    114,     # Syria
    738,     # Yemen
    351,     # north korea
    102850,  # Sudan
    297,     # Hezbollah
    153,     # hamas
]

# Pull closed events for each tag, deduplicate by event ID
all_events = {}

for tag_id in CONFLICT_TAGS:
    offset = 0
    while True:
        resp = requests.get(f"{GAMMA}/events", params={
            "tag_id": tag_id,
            "closed": "true",
            "limit": 100,
            "offset": offset,
        }, timeout=30)
        data = resp.json()
        if not data:
            break
        for e in data:
            eid = e.get("id")
            if eid not in all_events:
                all_events[eid] = e
        offset += 100
        if len(data) < 100:
            break
        time.sleep(0.2)
    
    print(f"tag_id={tag_id:<8d} -> total unique events so far: {len(all_events)}")
    time.sleep(0.2)

print(f"\nTotal unique closed conflict events: {len(all_events)}")

tag_id=100265   -> total unique events so far: 1375
tag_id=79       -> total unique events so far: 1388
tag_id=502      -> total unique events so far: 1396
tag_id=1346     -> total unique events so far: 1397
tag_id=464      -> total unique events so far: 1405
tag_id=193      -> total unique events so far: 1418
tag_id=593      -> total unique events so far: 1420
tag_id=863      -> total unique events so far: 1420
tag_id=1308     -> total unique events so far: 1421
tag_id=334      -> total unique events so far: 1432
tag_id=335      -> total unique events so far: 1432
tag_id=415      -> total unique events so far: 1434
tag_id=721      -> total unique events so far: 1434
tag_id=97       -> total unique events so far: 1436
tag_id=612      -> total unique events so far: 1436
tag_id=103027   -> total unique events so far: 1438
tag_id=102620   -> total unique events so far: 1438
tag_id=471      -> total unique events so far: 1440
tag_id=101409   -> total unique events so far: 1441
tag_id=69   

In [6]:
import json

# Extract all child markets with clear resolutions
markets = []

for eid, event in all_events.items():
    event_title = event.get("title", "")
    
    for m in event.get("markets", []):
        if not m.get("closed"):
            continue
        
        # Parse resolution
        try:
            outcomes = json.loads(m["outcomes"]) if isinstance(m["outcomes"], str) else m["outcomes"]
            prices = [float(p) for p in (json.loads(m["outcomePrices"]) if isinstance(m["outcomePrices"], str) else m["outcomePrices"])]
        except:
            continue
        
        resolution = None
        for i, p in enumerate(prices):
            if p > 0.95:
                resolution = outcomes[i]
                break
        if resolution is None:
            continue
        
        markets.append({
            "event_title": event_title,
            "question": m.get("question", ""),
            "slug": m.get("slug", ""),
            "resolution": resolution,
            "resolution_binary": 1 if resolution == "Yes" else 0,
            "start_date": m.get("startDateIso") or m.get("startDate"),
            "end_date": m.get("endDateIso") or m.get("endDate"),
            "volume": m.get("volumeNum", 0),
            "description": (m.get("description") or "")[:500],
        })

print(f"Total resolved markets (all geopolitics): {len(markets)}")

# Now filter to CONFLICT-ONLY via keywords
CONFLICT_KW = [
    "war", "invade", "invasion", "attack", "strike", "bomb", "shell",
    "missile", "military", "troops", "deploy", "nato",
    "ceasefire", "cease-fire", "truce", "armistice",
    "peace deal", "peace agreement", "peace treaty",
    "sanction", "embargo",
    "nuclear weapon", "nuclear strike",
    "coup", "overthrow",
    "conflict", "escalat", "retaliat",
    "shoot down", "airstrike", "air strike", "drone strike",
    "casualt", "killed in", "death toll",
    "territory", "captur", "annex", "occupied",
    "insurgent", "rebel", "militia",
    "houthi", "hezbollah", "hamas",
]

EXCLUDE_KW = [
    "super bowl", "nfl", "nba", "nhl", "mlb", "premier league",
    "oscar", "grammy", "emmy", "bitcoin", "ethereum", "solana",
    "price above", "price below", "market cap",
    "counter-strike", "strike price",  # gaming / finance
]

def is_conflict(question, description=""):
    text = (question + " " + description).lower()
    for ex in EXCLUDE_KW:
        if ex in text:
            return False
    return any(kw in text for kw in CONFLICT_KW)

conflict_markets = [m for m in markets if is_conflict(m["question"], m["description"])]
conflict_markets.sort(key=lambda x: x.get("volume", 0), reverse=True)

print(f"Conflict-only markets: {len(conflict_markets)}")
print(f"\nTop 30 by volume:")
for i, m in enumerate(conflict_markets[:30]):
    res = "YES" if m["resolution_binary"] == 1 else " NO"
    vol = f"${m['volume']:,.0f}" if m['volume'] else "?"
    print(f"{i+1:3d}. [{res}] {m['question'][:85]}")
    print(f"     Vol: {vol}")

Total resolved markets (all geopolitics): 4366
Conflict-only markets: 2676

Top 30 by volume:
  1. [ NO] Russia x Ukraine ceasefire in 2025?
     Vol: $73,752,993
  2. [ NO] Trump ends Ukraine war in first 90 days?
     Vol: $56,491,984
  3. [YES] Israel x Iran ceasefire before July? 
     Vol: $51,779,894
  4. [ NO] US x Venezuela military engagement by December 31?
     Vol: $51,073,021
  5. [YES] Yoon out as president of South Korea before May?
     Vol: $40,212,917
  6. [ NO] Yoon out as president of South Korea before April?
     Vol: $32,369,093
  7. [YES] U.S. anti-cartel ground operation in Mexico by January 31?
     Vol: $30,194,310
  8. [YES] US military action against Iran before July?
     Vol: $29,917,021
  9. [YES] Fordow nuclear facility destroyed before July?
     Vol: $28,623,187
 10. [YES] Israel military action against Iraq before November?
     Vol: $27,780,154
 11. [ NO] Will the US officially declare war on Iran before July?
     Vol: $18,143,784
 12. [YES] Major 

In [7]:
# Save full dataset
with open("resolved_conflict_polymarket.json", "w") as f:
    json.dump(conflict_markets, f, indent=2, default=str)

print(f"Saved {len(conflict_markets)} conflict markets")

# Quick stats
yes_count = sum(1 for m in conflict_markets if m["resolution_binary"] == 1)
no_count = len(conflict_markets) - yes_count
print(f"Resolved YES: {yes_count} ({100*yes_count/len(conflict_markets):.0f}%)")
print(f"Resolved NO:  {no_count} ({100*no_count/len(conflict_markets):.0f}%)")

Saved 2676 conflict markets
Resolved YES: 813 (30%)
Resolved NO:  1863 (70%)
